In [1]:
import numpy as np
import pandas as pd
import ollama
from datetime import datetime
from tqdm.auto import tqdm

# Load main dataset

In [2]:
# load dataframe
df_real = pd.read_csv('../data/data.csv', sep='\t', dtype=str)
df_real

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40944767,doi: 10.1007/s10803-025-07000-w,Psychometric Properties of the Social Responsi...,Journal of autism and developmental disorders,2025,"Fátima El-Bouhali-Abdellaoui, Núria Voltas, Pa...",Research Group on Nutrition and Mental Health ...,Earlier identification of autistic traits is c...
1,40944766,doi: 10.1007/s10803-025-07039-9,Exploring the Relationships Between Theory of ...,Journal of autism and developmental disorders,2025,"Jiaxi Li, Kathy Kar-Man Shum","Department of Psychology, The University of Ho...",This study examined friendship quality and the...
2,40944641,doi: 10.1002/hbm.70351,Flexible Reconfigurations of Brain Networks Du...,Human brain mapping,2025,"Qianying Wu, Zhihao Zhang, Ming Hsu, Andrew S ...","Helen Wills Neuroscience Institute, University...",How do large-scale brain networks dynamically ...
3,40944186,pii: 2798,Food Selectivity in Children with Autism Spect...,Nutrients,2025,"Paolo Mirizzi, Marco Esposito, Orlando Ricciar...",FUSIS MCF (Clinic Neuroscience Research and Tr...,Food selectivity is a prevalent and challengin...
4,40944170,pii: 2781,Gut Microbiota and Autism Spectrum Disorders: ...,Nutrients,2025,"Zuzanna Lewandowska-Pietruszka, Magdalena Figl...","Poznan University of Medical Sciences, Departm...",Autism spectrum disorder (ASD) is a complex ne...
...,...,...,...,...,...,...,...,...
2995,40250148,doi: 10.1016/j.yebeh.2025.110420,The prevalence of comorbidities in people with...,Epilepsy & behavior : E&B,2025,"Binx Yezhe Lin, Lisa Gong, Yifan Li, Hillary S...","Division of Addiction Science, Prevention, and...",To better understand medical comorbidity in pe...
2996,40249741,pii: e3003143,Vaccines work… and do not cause autism.,PLoS biology,2025,Nonia Pariente,"Public Library of Science, San Francisco, Cali...","Vaccines have saved millions of lives, yet the..."
2997,40249667,doi: 10.1080/17501911.2025.2491294,Mom genes and dad genes: genomic imprinting in...,Epigenomics,2025,"Erin M O'Leary, Paul J Bonthuis","Neuroscience Program, University of Illinois, ...",Genomic imprinting is an epigenetic phenomenon...
2998,40249409,doi: 10.1007/s10803-025-06815-x,Postural Control in Children with Autism Spect...,Journal of autism and developmental disorders,2025,"L Fradet, A Benchekri, R Tisserand, J-R Cazale...","Department of Child Psychiatry, Centre Hospita...",Autistic children (AT) are known to exhibit di...


# Sample PMIDs, initialize ground truth dataframe

In [3]:
# set sample size
sample_size = 100
print(sample_size)

100


In [4]:
# set number of questions per PMID
num_questions_per_pmid = 5
print(num_questions_per_pmid)

5


In [5]:
# sample PMIDs
missing_abstract = df_real['abstract'].isna()
sampled_pmids = df_real['pmid'][missing_abstract==False].sample(n=sample_size, \
random_state=824) # sample from those with abstract
sampled_pmids.values

array(['40348332', '40824395', '40663711', '40742673', '40454682',
       '40851977', '40369905', '40413384', '40890712', '40868529',
       '40460469', '40365849', '40806547', '40454245', '40255413',
       '40718817', '40324673', '40760909', '40625432', '40783757',
       '40414821', '40833659', '40828591', '40496977', '40612488',
       '40348275', '40481911', '40878890', '40782719', '40287553',
       '40715983', '40567447', '40471548', '40879460', '40295949',
       '40767168', '40404488', '40642101', '40465190', '40381091',
       '40331861', '40919359', '40852923', '40372603', '40704035',
       '40426452', '40435892', '40264093', '40485335', '40770710',
       '40770748', '40500588', '40766243', '40420478', '40629082',
       '40402836', '40905326', '40462172', '40686440', '40388040',
       '40556110', '40335664', '40823447', '40437788', '40691895',
       '40784411', '40392909', '40276113', '40635406', '40809815',
       '40924052', '40756590', '40497022', '40679751', '405210

In [6]:
# initialize ground truth dataframe
df_synth = pd.DataFrame({'pmid' : sorted(sampled_pmids.to_list()*num_questions_per_pmid), 
                      'ollama_seed' : [i for i in range(num_questions_per_pmid)]*sample_size})
df_synth = df_synth.merge(df_real, on=['pmid'], how='left')[['pmid', 'ollama_seed', 'abstract']]
df_synth # has seed for reproducibility

,pmid,ollama_seed,abstract
0,40255413,0,Gastrointestinal (GI) symptoms are frequently ...
1,40255413,1,Gastrointestinal (GI) symptoms are frequently ...
2,40255413,2,Gastrointestinal (GI) symptoms are frequently ...
3,40255413,3,Gastrointestinal (GI) symptoms are frequently ...
4,40255413,4,Gastrointestinal (GI) symptoms are frequently ...
...,...,...,...
495,40938164,0,The essay delineates a psychic phenomenon of i...
496,40938164,1,The essay delineates a psychic phenomenon of i...
497,40938164,2,The essay delineates a psychic phenomenon of i...
498,40938164,3,The essay delineates a psychic phenomenon of i...


# Use LLM to generate synthetic questions

In [7]:
# set LLM handle
model_handle = 'llama3.2:1b'
print(model_handle)

llama3.2:1b


In [8]:
# make prompt template for synthetic quesitons
prompt_template = """
You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}
""".strip()
print(prompt_template)

You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}


In [9]:
def generate_question(abstract, seed):
    prompt_text = prompt_template.format(abstract=abstract)
    response = ollama.chat(model=model_handle, messages=[{'role' : 'user', 'content' : prompt_text}],
                          options={'seed' : seed})
    return response['message']['content'].strip()

In [10]:
# demo question generation
demo_abstract = df_synth.iloc[0]['abstract']
print(demo_abstract)
print()
print(generate_question(demo_abstract, seed=42))

Gastrointestinal (GI) symptoms are frequently reported in children with autism spectrum disorder (ASD) and may significantly impact behavior, sleep, adaptive functioning, and the severity of autism. This study aims to explore the relationship between GI symptoms and these factors in children with ASD.

How do gastrointestinal symptoms affect the overall well-being and daily life of individuals on the autism spectrum?


In [11]:
def generate_questions_list(df):
    records = df.to_dict(orient='records')
    synthetic_questions = [generate_question(record['abstract'], record['ollama_seed']) \
    for record in tqdm(records)]
    return synthetic_questions

In [12]:
# generate synthetic
print(datetime.now())
df_synth['synthetic_question'] = generate_questions_list(df_synth)
print(datetime.now())

2025-09-14 00:37:15.755032


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-14 00:53:00.351239


In [13]:
# to demonstrate reproducibility, generate again
if False: # set to True to generate again and compare
    print(datetime.now())
    demo_reproducibility = pd.Series(generate_questions_list(df_synth))
    print(datetime.now())
    print((df_synth['synthetic_question']==demo_reproducibility).value_counts()) # all true if reproducible
    print(pd.concat([df_synth['synthetic_question'], demo_reproducibility], axis=1))

In [14]:
# finalize ground truth dataframe
df_synth = df_synth[['pmid', 'ollama_seed', 'synthetic_question']]
df_synth

,pmid,ollama_seed,synthetic_question
0,40255413,0,Is gastrointestinal (GI) problems more common ...
1,40255413,1,Does autism spectrum disorders affect a child'...
2,40255413,2,What is one possible underlying factor contrib...
3,40255413,3,What is a common challenge that many individua...
4,40255413,4,How do gastrointestinal (GI) symptoms affect t...
...,...,...,...
495,40938164,0,Is autism a natural part of human diversity?
496,40938164,1,Can individuals with autism spectrum disorders...
497,40938164,2,What is it about individuals with autism spect...
498,40938164,3,What is the main difference between autism and...


In [15]:
# look at a few questions
print('\n'.join(df_synth['synthetic_question'].to_list()[:10]))

Is gastrointestinal (GI) problems more common in individuals with autism spectrum disorders than in the general population?
Does autism spectrum disorders affect a child's ability to cope with nighttime feedings?
What is one possible underlying factor contributing to the co-occurrence of gastrointestinal (GI) symptoms in children with autism spectrum disorder (ASD)?
What is a common challenge that many individuals with autism spectrum disorders face when it comes to maintaining regular physical activity?
How do gastrointestinal (GI) symptoms affect the daily lives and behaviors of individuals with autism spectrum disorder?
Is there a link between autism and difficulties with intimacy or relationships?
How can individuals on the autism spectrum best navigate their sexuality?
Is Autism more likely to affect a person's ability to form healthy romantic relationships?
Is there a difference between the prevalence of autism spectrum disorder and the prevalence of other conditions that can als

# Write CSV

In [16]:
# write CSV file
df_synth.to_csv('../data/data-synth-question.csv', index=False, sep='\t')

In [17]:
print(datetime.now())

2025-09-14 00:53:00.383952
